### Youtube Chatbot 
A RAG based application that will let you ask questions from any youtube video which have a transcript

In [48]:
from dotenv import load_dotenv
load_dotenv()

True

## 1. Data Gathering


In [19]:
from youtube_transcript_api import YouTubeTranscriptApi, FetchedTranscript
import os

In [ ]:
url = "https://www.youtube.com/watch?v=Iwpi1Lm6dFo"
def get_id(url:str) -> str:
    id = url.split("v=")[-1].split("&t")[0]
    return id

id = get_id(url=url)
transcripter = YouTubeTranscriptApi()
transcript = transcripter.fetch(video_id=id, languages=['en'])  #This returns a "FetchedTranscript" (iterable) object which contains list of "FetchedTranscriptSnippet"
 
for lines in transcript:
    print(lines, end="\n")

FetchedTranscriptSnippet(text='Translator: ', start=0.0, duration=7.0)
FetchedTranscriptSnippet(text='Okay, ladies and gentlemen, welcome.', start=11.011, duration=2.317)
FetchedTranscriptSnippet(text='There is a question', start=14.488, duration=1.148)
FetchedTranscriptSnippet(text='which has puzzled me for quite a while,', start=15.636, duration=2.175)
FetchedTranscriptSnippet(text='and that is, Why do our PowerPoints\nlook the way they look?', start=18.261, duration=3.277)
FetchedTranscriptSnippet(text='Or rather, How on earth, can we accept\nthat they look the way they look?', start=22.228, duration=4.75)
FetchedTranscriptSnippet(text='How can you do that?', start=26.978, duration=1.701)
FetchedTranscriptSnippet(text='And do you know', start=29.249, duration=1.016)
FetchedTranscriptSnippet(text="what's even more intellectually\nchallenging for me to understand?", start=30.265, duration=3.107)
FetchedTranscriptSnippet(text="It's, How can a person sit over here\nin this meeting room 

In [18]:
def transcript_to_text(transcript:FetchedTranscript) -> str:
    text = ""
    for snippet in transcript: 
        text += snippet.text + " "
    return text

content = transcript_to_text(transcript)

In [20]:
## Store this in a .txt file
os.makedirs(name = "Transcripts", exist_ok=True)
with open("Transcripts/transcript.txt", 'w') as file:
    file.write(content)

## 2. Document Loading

In [21]:
from langchain_community.document_loaders import TextLoader

In [37]:
loader = TextLoader(file_path = "Transcripts/transcript.txt")
doc = loader.load()

## 3. Text Splitting

In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [43]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)
text_splits = text_splitter.split_documents(doc)

print(len(text_splits))

43


we splitted the whole text into 43 chunks 

## Vectorization of Chunks and Storage

In [49]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings, HuggingFaceEmbeddings
from langchain_chroma import Chroma 

In [53]:
embedding_llm = HuggingFaceEndpointEmbeddings(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    task = 'feature-extraction'
)

vector_strore = Chroma(
    embedding_function=embedding_llm,
    persist_directory="Chroma_VectorStore",
    collection_name="transcript"
)

In [55]:
vector_strore.add_documents(text_splits)

['8d8d69da-d365-4111-84ad-b7e25f44faa1',
 '310a54ba-4c60-4deb-8898-fa80e8c67f9b',
 '38ec2e2b-2dde-4e90-857f-fb0825c6c7af',
 '29fe7ec1-02c9-45ab-9095-0e6cb6573462',
 '2f093d35-08b9-4093-8977-8588e2f86e45',
 '23aa090a-17d8-4b2d-a4e7-de5982fc2112',
 '3dd3865c-911c-430f-9e58-9752e4542a4b',
 '617aaefc-b8b5-47ee-b31e-a00182860ca5',
 'd55dc0df-7285-43f6-b966-f58a95f6ceba',
 '7f7b0852-fb3c-4f60-838e-cabc4f488d6c',
 'e0bebb65-b05b-4725-bf16-af346a1dd8ad',
 '0a32e46d-cc3a-48c7-82e5-64e80af57c6c',
 'fa3e69e0-8fe3-4d48-897a-b92bab451be1',
 'f299f402-3007-455f-af10-50b799e5014a',
 '08cbc15f-9578-4e7d-8a18-cbac9bd32ab5',
 'b1f83db0-b8f4-4f6a-9644-41cdf6333745',
 '7084f482-b1c1-4ab3-8d33-3e5daf6c4106',
 '4fe3af33-d630-4c7b-83a1-c90c3e6fb7da',
 '7d9eaa63-50d2-4bf6-bf4e-dfb152715f34',
 '3cfb9b65-0fa7-440b-b900-e1ff320be93e',
 '4e98282e-e915-4ef1-ada2-fcdbac46030d',
 '8154c1a2-28d7-4cfe-a07e-828c200db213',
 '56fda783-7fd2-48c3-a641-6c1f87de8d0f',
 '8b43343f-9db0-4e23-a238-aa7b82abe89f',
 'd8b6c8be-e01c-

## 5. Retrieval

In [104]:
retriever = vector_strore.as_retriever(search_type = "similarity", search_kwargs={'k':5})
retrieved = retriever.invoke("What is most important thing to consider in powerpoint?")
retrieved

[Document(id='0a32e46d-cc3a-48c7-82e5-64e80af57c6c', metadata={'source': 'Transcripts/transcript.txt'}, page_content="called John Medina, and he puts it like this. [If companies would have\nas little respect for business as they have for presentations, the majority would go bankrupt.] And it's with his words that I welcome you\nto 'How to Avoid Death by PowerPoint'. Now, my objective for this evening,\nfor these 18 minutes, is to give you five design principles that will cognitively and psychologically"),
 Document(id='4e98282e-e915-4ef1-ada2-fcdbac46030d', metadata={'source': 'Transcripts/transcript.txt'}, page_content="where is your attention drawn to without you even having a chance\nof controlling it? It's going to the big three all the time. Have a look at the practical situation;\nhave a look at this. Where are your eyes drawn to? I can see that they're drawn\nconstantly to the headline. Now, how often is the headline\nthe most important part in a PowerPoint? It's very rare. Even

## 6. Augmentation

In [60]:
from langchain_core.prompts import PromptTemplate

In [115]:
template = PromptTemplate(
    input_variables=["transcript", "query"],
    
    template="""
You are a helpful YouTube video assistant.

Your task is to answer the user's question using the provided video transcript.

Relevant part of Transcript:
{transcript}

User Question:
{query}

Instructions:
1. Answer the question clearly and concisely using the transcript.
2. If the transcript contains partial information, mention that explicitly.
3. If the transcript does not contain enough information to answer the question:
   - First say: "There is not enough context in the video transcript."
   - Then provide a general answer based on your overall knowledge.
   - Clearly mention that the answer is a general explanation and may not reflect the video's actual content.
4. Do not make up details that are not present in the transcript.
"""
)

## 7. Generation 

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [67]:
gen_llm = HuggingFaceEndpoint(
    repo_id = "deepseek-ai/DeepSeek-V4-Flash",
    task='text-generation'
)

chat_model = ChatHuggingFace(llm=gen_llm)

## Final: Chaining

In [107]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [106]:
##Format docs first
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

In [133]:
parser = StrOutputParser()
parallel_chain = RunnableParallel(
    {
        'transcript': retriever | RunnableLambda(format_docs) | parser,
        'query': RunnablePassthrough()
    }
)

final_chain = parallel_chain | template | chat_model | parser

In [134]:
print(final_chain.get_graph().draw_ascii())

             +---------------------------------+          
             | Parallel<transcript,query>Input |          
             +---------------------------------+          
                    ***               ***                 
                 ***                     ***              
               **                           ***           
+----------------------+                       **         
| VectorStoreRetriever |                        *         
+----------------------+                        *         
            *                                   *         
            *                                   *         
            *                                   *         
    +-------------+                             *         
    | format_docs |                             *         
    +-------------+                             *         
            *                                   *         
            *                                   *       

## Test

In [117]:
result = final_chain.invoke("How many slides should a ppt have?")

In [119]:
print(result)

 solely on the provided transcript, there is not enough context to answer the question about how many slides a presentation should have.

The transcript does not specify a number of slides. Instead, it criticizes the idea of limiting slides (e.g., "You can only use four") and focuses on the problem being the **amount of objects per slide**, not the number of slides. The speaker argues that having more slides is better than cramming too much content onto a few slides, but no specific recommended number of slides is given.
